# EAV EEG / AVE Experiments

目标：在 EAV 预提取特征上接入 EEG，依次跑三组实验：

```text
1. Temporal EEG
2. Temporal AVE
3. QueryToken AVE
```

这三组放在同一个 notebook 里是合理的：它们共用同一份数据加载、评估函数和训练逻辑，方便横向对比。等结果稳定后，再整理成项目脚本。

当前关键对照：

| 方法 | Accuracy | Macro-F1 |
|---|---:|---:|
| Temporal AV | 0.7355 | 0.7368 |
| QueryToken AV + AVCL, q=16 | 0.7486 | 0.7470 |

判断 EEG 是否有增益，重点看：

```text
Temporal AVE 是否超过 Temporal AV
QueryToken AVE 是否超过 QueryToken AV + AVCL
```


## 1. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 解压 EAV 数据

如果 `/content/eav_data` 已经存在且数据完整，可以跳过解压 cell。Colab 断开后本地 `/content` 会清空，需要重新解压。

In [ ]:
import os

ARCHIVE_PATH = "/content/drive/MyDrive/EAV/mind_eav.tar.zst"
EXTRACT_DIR = "/content/eav_data"

print("archive exists:", os.path.exists(ARCHIVE_PATH))
print("extract dir exists:", os.path.exists(EXTRACT_DIR))


In [ ]:
# 如果 /content/eav_data 不存在，运行本 cell 解压。
!apt-get -qq update
!apt-get -qq install -y zstd
!mkdir -p /content/eav_data
!tar --use-compress-program=unzstd -xf "/content/drive/MyDrive/EAV/mind_eav.tar.zst" -C /content/eav_data


## 3. 自动查找 EAV_ROOT

In [ ]:
import os

EAV_ROOT = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    if "sub01" in dirs and "demographics.csv" in files:
        EAV_ROOT = root
        break

print("EAV_ROOT =", EAV_ROOT)
assert EAV_ROOT is not None, "未找到 EAV_ROOT，请检查解压目录。"
print(os.listdir(EAV_ROOT)[:10])


## 4. 导入依赖与全局配置

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = Path("/content/drive/MyDrive/EAV/results_eeg_ave")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = ["Neutral", "Sadness", "Anger", "Happiness", "Calmness"]

print("device:", DEVICE)
print("save dir:", SAVE_DIR)
print("EAV_ROOT:", EAV_ROOT)


## 5. 检查数据形状

In [ ]:
sub_dir = os.path.join(EAV_ROOT, "sub01")

for name in ["audio.npy", "video.npy", "eeg.npy", "labels.npy", "split.npy"]:
    arr = np.load(os.path.join(sub_dir, name), mmap_mode="r")
    print(name, arr.shape, arr.dtype)


## 6. Dataset 与 DataLoader

当前读取三种模态：

```text
audio: (501, 256)
video: (25, 512)
eeg:   (500, 30)
```

EEG 会在模型 forward 中做 per-sample z-score 标准化。

In [ ]:
class EAVAVEDataset(Dataset):
    def __init__(self, root, train=True):
        self.items = []
        target_split = 0 if train else 1

        subjects = sorted([d for d in os.listdir(root) if d.startswith("sub")])
        for sub in subjects:
            sub_dir = os.path.join(root, sub)
            labels = np.load(os.path.join(sub_dir, "labels.npy"), mmap_mode="r")
            split = np.load(os.path.join(sub_dir, "split.npy"), mmap_mode="r")

            indices = np.where(split == target_split)[0]
            for i in indices:
                self.items.append((sub_dir, int(i), int(labels[i])))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        sub_dir, i, label = self.items[idx]
        audio = np.load(os.path.join(sub_dir, "audio.npy"), mmap_mode="r")[i]
        video = np.load(os.path.join(sub_dir, "video.npy"), mmap_mode="r")[i]
        eeg = np.load(os.path.join(sub_dir, "eeg.npy"), mmap_mode="r")[i]

        audio = torch.from_numpy(np.asarray(audio, dtype=np.float32))
        video = torch.from_numpy(np.asarray(video, dtype=np.float32))
        eeg = torch.from_numpy(np.asarray(eeg, dtype=np.float32))
        label = torch.tensor(label, dtype=torch.long)
        return audio, video, eeg, label


def make_loaders(batch_size=64, num_workers=0):
    train_ds = EAVAVEDataset(EAV_ROOT, train=True)
    test_ds = EAVAVEDataset(EAV_ROOT, train=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    return train_ds, test_ds, train_loader, test_loader

train_ds = EAVAVEDataset(EAV_ROOT, train=True)
test_ds = EAVAVEDataset(EAV_ROOT, train=False)
print("train:", len(train_ds))
print("test:", len(test_ds))
a, v, e, y = train_ds[0]
print("sample audio:", a.shape, "video:", v.shape, "eeg:", e.shape, "label:", y)


## 7. 通用模块

`TemporalEncoder` 用于 Temporal EEG / Temporal AVE：

```text
sequence -> Linear projection -> attention pooling -> representation
```

`QueryCrossAttentionBlock` 用于 QueryToken AVE。

In [ ]:
def normalize_eeg(eeg):
    # eeg: (B, T, C)，按每个样本的时间维做 z-score，降低幅值尺度影响。
    mean = eeg.mean(dim=1, keepdim=True)
    std = eeg.std(dim=1, keepdim=True).clamp(min=1e-6)
    return (eeg - mean) / std


class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.score = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.score(x), dim=1)
        return (x * weights).sum(dim=1)


class TemporalEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.pool = AttentionPooling(hidden_dim)

    def forward(self, x):
        x = self.proj(x)
        return self.pool(x)


class QueryCrossAttentionBlock(nn.Module):
    def __init__(self, hidden_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.query_self_attn = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout),
        )
        self.norm_q1 = nn.LayerNorm(hidden_dim)
        self.norm_q2 = nn.LayerNorm(hidden_dim)
        self.norm_ffn = nn.LayerNorm(hidden_dim)

    def forward(self, query_tokens, sequence):
        q = self.norm_q1(query_tokens)
        q_self, _ = self.query_self_attn(q, q, q, need_weights=False)
        query_tokens = query_tokens + q_self

        q = self.norm_q2(query_tokens)
        q_cross, _ = self.cross_attn(q, sequence, sequence, need_weights=False)
        query_tokens = query_tokens + q_cross

        query_tokens = query_tokens + self.ffn(self.norm_ffn(query_tokens))
        return query_tokens


## 8. 模型 1：Temporal EEG

In [ ]:
class TemporalEEGClassifier(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.1, num_classes=5):
        super().__init__()
        self.eeg_encoder = TemporalEncoder(30, hidden_dim, dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, audio, video, eeg):
        eeg = normalize_eeg(eeg)
        feature = self.eeg_encoder(eeg)
        return self.classifier(feature)


## 9. 模型 2：Temporal AVE

In [ ]:
class TemporalAVEClassifier(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.1, num_classes=5):
        super().__init__()
        self.audio_encoder = TemporalEncoder(256, hidden_dim, dropout)
        self.video_encoder = TemporalEncoder(512, hidden_dim, dropout)
        self.eeg_encoder = TemporalEncoder(30, hidden_dim, dropout)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, audio, video, eeg):
        eeg = normalize_eeg(eeg)
        audio_repr = self.audio_encoder(audio)
        video_repr = self.video_encoder(video)
        eeg_repr = self.eeg_encoder(eeg)
        feature = self.fusion(torch.cat([audio_repr, video_repr, eeg_repr], dim=1))
        return self.classifier(feature)


## 10. 模型 3：QueryToken AVE

In [ ]:
class QueryTokenAVEClassifier(nn.Module):
    def __init__(
        self,
        hidden_dim=256,
        num_query=16,
        num_heads=4,
        num_layers=2,
        dropout=0.1,
        num_classes=5,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_query = num_query

        self.audio_proj = nn.Sequential(nn.Linear(256, hidden_dim), nn.LayerNorm(hidden_dim), nn.Dropout(dropout))
        self.video_proj = nn.Sequential(nn.Linear(512, hidden_dim), nn.LayerNorm(hidden_dim), nn.Dropout(dropout))
        self.eeg_proj = nn.Sequential(nn.Linear(30, hidden_dim), nn.LayerNorm(hidden_dim), nn.Dropout(dropout))

        self.audio_query_tokens = nn.Parameter(torch.zeros(1, num_query, hidden_dim))
        self.video_query_tokens = nn.Parameter(torch.zeros(1, num_query, hidden_dim))
        self.eeg_query_tokens = nn.Parameter(torch.zeros(1, num_query, hidden_dim))
        nn.init.normal_(self.audio_query_tokens, mean=0.0, std=0.02)
        nn.init.normal_(self.video_query_tokens, mean=0.0, std=0.02)
        nn.init.normal_(self.eeg_query_tokens, mean=0.0, std=0.02)

        self.audio_blocks = nn.ModuleList([
            QueryCrossAttentionBlock(hidden_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.video_blocks = nn.ModuleList([
            QueryCrossAttentionBlock(hidden_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.eeg_blocks = nn.ModuleList([
            QueryCrossAttentionBlock(hidden_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def encode_modality(self, sequence, query_tokens, blocks):
        batch_size = sequence.size(0)
        queries = query_tokens.expand(batch_size, -1, -1)
        for block in blocks:
            queries = block(queries, sequence)
        return queries.mean(dim=1)

    def forward(self, audio, video, eeg):
        eeg = normalize_eeg(eeg)
        audio_seq = self.audio_proj(audio)
        video_seq = self.video_proj(video)
        eeg_seq = self.eeg_proj(eeg)

        audio_repr = self.encode_modality(audio_seq, self.audio_query_tokens, self.audio_blocks)
        video_repr = self.encode_modality(video_seq, self.video_query_tokens, self.video_blocks)
        eeg_repr = self.encode_modality(eeg_seq, self.eeg_query_tokens, self.eeg_blocks)

        feature = self.fusion(torch.cat([audio_repr, video_repr, eeg_repr], dim=1))
        return self.classifier(feature)


## 11. 训练与评估函数

In [ ]:
def evaluate(model, test_loader):
    model.eval()
    preds, golds = [], []

    with torch.no_grad():
        for audio, video, eeg, labels in test_loader:
            audio = audio.to(DEVICE, non_blocking=True)
            video = video.to(DEVICE, non_blocking=True)
            eeg = eeg.to(DEVICE, non_blocking=True)

            logits = model(audio, video, eeg)
            pred = logits.argmax(dim=1).cpu().numpy()
            preds.extend(pred)
            golds.extend(labels.numpy())

    acc = accuracy_score(golds, preds)
    macro_f1 = f1_score(golds, preds, average="macro")
    report = classification_report(golds, preds, target_names=LABEL_NAMES)
    return acc, macro_f1, report


def train_model(
    model,
    run_name,
    epochs=30,
    batch_size=64,
    lr=1e-4,
    num_workers=0,
):
    train_ds, test_ds, train_loader, test_loader = make_loaders(batch_size, num_workers)
    print("train:", len(train_ds), "test:", len(test_ds))
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    ce_loss = nn.CrossEntropyLoss()
    best = {"epoch": 0, "acc": 0.0, "macro_f1": 0.0, "report": None}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for audio, video, eeg, labels in train_loader:
            audio = audio.to(DEVICE, non_blocking=True)
            video = video.to(DEVICE, non_blocking=True)
            eeg = eeg.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(audio, video, eeg)
            loss = ce_loss(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)

        acc, macro_f1, report = evaluate(model, test_loader)
        avg_loss = total_loss / len(train_ds)
        print(f"epoch={epoch:02d} loss={avg_loss:.4f} acc={acc:.4f} macro_f1={macro_f1:.4f}")

        if macro_f1 > best["macro_f1"]:
            best["epoch"] = epoch
            best["acc"] = acc
            best["macro_f1"] = macro_f1
            best["report"] = report
            torch.save(
                {"epoch": epoch, "model": model.state_dict(), "run_name": run_name},
                SAVE_DIR / f"best_{run_name}.pt",
            )

    result = {
        "model": run_name,
        "best_epoch": best["epoch"],
        "best_acc": best["acc"],
        "best_macro_f1": best["macro_f1"],
    }
    result_path = SAVE_DIR / f"result_{run_name}.json"
    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)

    print("
Best result:")
    print(result)
    print(best["report"])
    print("saved to:", result_path)
    return best


## 12. 实验 1：Temporal EEG

In [ ]:
temporal_eeg_result = train_model(
    model=TemporalEEGClassifier(hidden_dim=256, dropout=0.1),
    run_name="temporal_eeg",
    epochs=30,
    batch_size=64,
    lr=1e-4,
    num_workers=0,
)


## 13. 实验 2：Temporal AVE

In [ ]:
temporal_ave_result = train_model(
    model=TemporalAVEClassifier(hidden_dim=256, dropout=0.1),
    run_name="temporal_ave",
    epochs=30,
    batch_size=64,
    lr=1e-4,
    num_workers=0,
)


## 14. 实验 3：QueryToken AVE

In [ ]:
query_ave_result = train_model(
    model=QueryTokenAVEClassifier(
        hidden_dim=256,
        num_query=16,
        num_heads=4,
        num_layers=2,
        dropout=0.1,
    ),
    run_name="query_ave_q16_l2_ce",
    epochs=30,
    batch_size=64,
    lr=1e-4,
    num_workers=0,
)


## 15. 查看已保存结果

In [ ]:
for path in sorted(SAVE_DIR.glob("result_*.json")):
    with open(path, "r") as f:
        result = json.load(f)
    print(path.name, result)
